### Step 1: Import Libraries & API Keys

In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display
import gradio as gr
import json
import requests
import re
from typing import List

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API key is missing.")

client = OpenAI(api_key=OPENAI_API_KEY)

### Step 2: RAG Preparation

In [ ]:
document_overview = """
Mario Anibal Cruz Hernandez, known online as Ermac517, is a Senior DevOps and Infrastructure Engineer residing in the Houston and Pearland area of Texas. 
He holds a Master’s degree, for which he self-taught the C programming language to build a secured DNS Proxy Server for his thesis. 
His professional tenure includes a long-standing career at the software company PROS, which he joined in July 2011 after completing a 
rigorous eight-round interview process. Specializing in cloud-native infrastructure, Mario possesses deep technical expertise across Microsoft Azure, AWS, and 
Google Cloud Platform, with a particular focus on production migrations, CI/CD pipelines, containerization, and Apache Cassandra database systems. 
Committed to continuous growth, he earned a Data Engineer certification in March 2026 and is currently pivoting his career toward AI Engineering and 
MLOps through advanced training in Agentic AI.

Beyond his engineering career, Mario is an exceptionally dedicated PlayStation trophy hunter, a pursuit he began in 2008 and has since resulted 
in an impressive collection of more than 12,000 trophies. His favorite gaming franchises lean toward action and horror, including Resident Evil, 
God of War, and Metal Gear. His entertainment interests are wide-ranging; he has been a dedicated follower of professional wrestling since the 1990s—closely 
tracking WWE events and WrestleMania statistics—and follows K-pop culture, film, and television. He also applies his analytical skills to sports, 
utilizing data-driven statistical modeling for fantasy leagues and sports betting centered on the Premier League and La Liga.

In his personal life, Mario is married to Dulce Jimenez, whose career guidance he considers the most influential advice he has 
received, and he enjoys sharing family-friendly hobbies and activities with his children. He manages a modern, automated home equipped with 
smart security and vehicle safety tech. His appreciation for nature is evident in his backyard, where he has researched the behavioral and nesting 
habits of wild rabbits to build a welcoming habitat for them. When ordering food, he has a strong preference for thin-crust pizzas from Domino's and Pizza Hut,
alongside an appreciation for Mexican, Asian, and Mediterranean cuisines. He is also a frequent traveler, with recent journeys taking him across various regions of 
Mexico, including Monterrey, Mexico City, Guadalajara, and Tuxtla Gutiérrez.

Additonal info:
- In 1999, Mario graduated from high school and started attending college.
- His favorite video games include Mortal Kombat, Street Fighter, The King of Fighters, and Killzone.
- His favorite movies include Star Wars, Lord of the Rings, The Matrix, Marvel Cinematic Universe, and Batman.
- He has owned several gaming consoles over the years, including Playstation 2, Playstation 3, Playstation 4, Playstation 5, Gamecube, Wii, and Switch.
- He currently has no friends.
"""

In [ ]:
document_education = """
Education

Tecnológico de Monterrey logo
Tecnológico de Monterrey

Master's degree, Information Technology

2005 – 2007

Grade: 98/100

Thesis project — Designed and implemented a C-based DNS proxy that performs DNSSEC “online signing” to prevent zone‑enumeration attacks; integrated OpenSSL and other open‑source libraries and validated on Linux and FreeBSD.… more

Tecnológico de Monterrey logo
Tecnológico de Monterrey

Bachelor's Degree, Systems Engineering

1999 – 2003

Grade: 96/100
"""

In [ ]:
document_professional_experience = """
PROS logo
PROS

Full-time · 14 yrs 11 mos

Principal Software Engineer

Jul 2023 - Present · 3 yrs

Houston, Texas, United States · Hybrid

• Led the migration of on-premises environments to cloud platforms, ensuring zero downtime throughout the process.
• Designed and implemented Azure DevOps / GitHub Actions pipelines, enabling developers to deliver releases more rapidly and efficiently.
• Developed and integrated comprehensive monitoring frameworks and tools across all cloud environments, enabling rapid issue detection and proactive outage prevention.

Skills: Apache Spark, Agile Methodologies, +29 skills

Senior Software Engineer

Mar 2015 - Jun 2023 · 8 yrs 4 mos

Greater Houston · On-site

• Architected and developed highly available, mission-critical Java applications, delivering exceptional reliability and performance for end users.
• Optimized application build scripts to reduce runtime and deliver clear, accurate test results to developers and QA engineers, increasing overall team efficiency.
• Developed and deployed NoSQL interface modules and APIs, enabling seamless integration of databases such as Cassandra into applications with minimal code changes, streamlining the development process for engineers.

Skills: Agile Methodologies, Chef, +26 skills

Software Engineer II

Aug 2011 - Feb 2015 · 3 yrs 7 mos

Greater Houston · On-site

• Implemented REST frameworks to build Java microservices for the airline travel sector, ensuring optimal performance under heavy load and full compliance with stringent SLAs.
• Implemented Cassandra APIs to facilitate a successful migration from Berkeley DB to Cassandra, resulting in over a 50% improvement in performance.
• Successfully transitioned from legacy Maven build scripts to Gradle, optimizing build times and enabling integration with Jenkins for automated workflows.

Skills: Agile Methodologies, Gradle, +22 skills

NIC Mexico logo
NIC Mexico

Full-time · 3 yrs 11 mos

Monterrey, Nuevo León, Mexico · On-site

System Administrator

Jun 2008 - May 2011 · 3 yrs

- Implemented automated monitoring scripts for NIC Mexico’s MX Registry and Registrar, enabling real-time health checks and proactive alerting.
- Led zero‑downtime data center migrations and hardware upgrades for the .MX top‑level domain, ensuring uninterrupted DNS resolution.

Skills: Bash, Java, +7 skills

Software Engineer

Jul 2007 - May 2008 · 11 mos

Designed and implemented a DNS proxy in C that performs online signing to mitigate zone‑enumeration attacks.

Skills: JavaScript, SQL, +9 skills

Tecnológico de Monterrey logo
Research Assistant

Tecnológico de Monterrey · Internship

Aug 2005 - May 2007 · 1 yr 10 mos

Monterrey, Nuevo León, Mexico · On-site

Developed a security-focused proxy to resolve DNSSEC zone enumeration flaws. Integrated the "White Lies" protocol to prevent domain disclosure, ensuring server privacy and protecting against bulk data harvesting by attackers.

 JavaScript, SQL and +10 skills

Softtek logo
Software Engineer

Softtek · Full-time

Jan 2004 - Jul 2005 · 1 yr 7 mos

Monterrey, Nuevo León, Mexico · On-site

Engineered secure J2EE backend modules to tighten access control for critical turbine and generator configuration systems, reducing unauthorized or redundant access by 40%. Stack included IBM WebSphere, Tomcat, Oracle.

 JavaScript, SQL and +8 skills
"""

In [ ]:
# Chunk the document
def split_text_into_chunks(
    text: str,
    chunk_size: int = 200,
    overlap: int = 50,
) -> List[str]:
    """
    Split text into overlapping chunks, preferring natural boundaries.

    Rules:
    - Each chunk is at most `chunk_size` characters.
    - Consecutive chunks overlap by `overlap` characters.
    - If the chunk would end mid-sentence/paragraph, move the cut backward
      to the best natural boundary, in this order:
        1. paragraph break
        2. newline
        3. sentence end
        4. space
    - Only move the cut backward if the boundary is after the halfway point
      of the candidate chunk.

    Returns:
        List[str]: list of text chunks.
    """
    if not text:
        return []

    if chunk_size <= 0:
        raise ValueError("chunk_size must be > 0")
    if overlap < 0:
        raise ValueError("overlap must be >= 0")
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    def find_sentence_boundary(s: str, min_pos: int) -> int:
        # Match sentence-ending punctuation optionally followed by quotes/brackets,
        # then whitespace or end of string.
        matches = list(re.finditer(r'[.!?][\'")\]]*(?=\s|$)', s))
        for match in reversed(matches):
            end = match.end()
            if end >= min_pos:
                return end
        return -1

    chunks = []
    start = 0
    n = len(text)

    while start < n:
        candidate_end = min(start + chunk_size, n)

        if candidate_end == n:
            end = n
        else:
            window = text[start:candidate_end]
            half_point = len(window) // 2

            end = -1

            # 1. Paragraph break
            idx = window.rfind("\n\n", half_point)
            if idx != -1:
                end = start + idx + 2

            # 2. Newline
            if end == -1:
                idx = window.rfind("\n", half_point)
                if idx != -1:
                    end = start + idx + 1

            # 3. Sentence end
            if end == -1:
                idx = find_sentence_boundary(window, half_point)
                if idx != -1:
                    end = start + idx

            # 4. Space
            if end == -1:
                idx = window.rfind(" ", half_point)
                if idx != -1:
                    end = start + idx + 1

            # Fallback: hard cut
            if end == -1 or end <= start:
                end = candidate_end

        chunk = text[start:end]
        if chunk:
            chunks.append(chunk)

        if end >= n:
            break

        start = max(end - overlap, start + 1)

    return chunks

In [ ]:


documents = [
    {"text":document_overview, "source": "Overview"},
    {"text":document_professional_experience, "source": "Professional Experience"},
    {"text":document_education, "source": "Education"}
]

chunks = []
ids = []
metadatas = []

for doc in documents:
    # Prepare the lists
    chunks_ = split_text_into_chunks(doc["text"], chunk_size=300, overlap=50)
    ids_ = [str(uuid.uuid4()) for _ in range(len(chunks_))]
    metadatas_ = [{"source": doc["source"], "chunk_index": i} for i in range(len(chunks_))] 
    
    # Add to main lists
    chunks.extend(chunks_)
    ids.extend(ids_)
    metadatas.extend(metadatas_)

print(f"Total chunks: {len(chunks)}\n")

for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1} (ID: {ids[i]}, Source: {metadatas[i]['source']}, Index: {metadatas[i]['chunk_index']}, Length: {len(chunk)}):")
    print(chunk)
    print()

In [ ]:
# Generate embeddings for all chunks
response = client.embeddings.create(
    model = "text-embedding-3-small",
    input = chunks
)

embeddings = [item.embedding for item in response.data]

In [ ]:
# Print the raw response data for debugging

pprint(response.data)

# Verify embeddings
print(f"Total embeddings: {len(embeddings)}\n")
print(f"Embedding dimension: {len(embeddings[0])}\n")

In [ ]:
# Graph the embeddings using t-SNE and color by KMeans clusters

import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans

X = np.array(embeddings)
clusters = KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(X)

color_map = {0: "red", 1: "blue"}
point_colors = [color_map[c] for c in clusters]

p = min(30, len(X) - 1)
X2 = TSNE(n_components=2, perplexity=p, random_state=42).fit_transform(X)

plt.figure(figsize=(10, 8))
plt.scatter(X2[:, 0], X2[:, 1], c=point_colors, s=40, alpha=0.8)

for i, (x, y) in enumerate(X2, start=1):
    plt.text(x, y, str(i), fontsize=8)

plt.title("t-SNE of Embeddings (2 Clusters)")
plt.xlabel("Dim 1")
plt.ylabel("Dim 2")
plt.show()

grouped = {0: [], 1: []}
for i, (label, chunk) in enumerate(zip(clusters, chunks), start=1):
    grouped[label].append((i, chunk))

def describe_cluster(items, cluster_id, color):
    sample = "\n\n".join(
        f"Chunk {i}:\n{chunk[:1000]}" for i, chunk in items[:8]
    )

    response = client.responses.create(
        model="gpt-4.1-mini",
        input=f"""
You are analyzing a cluster of text chunks grouped by embedding similarity.

Describe this cluster in 2-4 sentences.
Start the response with:
Cluster {cluster_id} ({color})

Include:
- the main theme
- what makes these chunks similar
- any notable subtopic or style pattern

Here are sample chunks from Cluster {cluster_id}:

{sample}
"""
    )
    return response.output_text

for cluster_id, items in grouped.items():
    color = color_map[cluster_id]
    print(f"\n=== Cluster {cluster_id} ({color}, {len(items)} chunks) ===")
    print("Chunk numbers:", [i for i, _ in items])
    print(describe_cluster(items, cluster_id, color))

In [ ]:
# Initialize ChromaDB client and collection

# Initialize ChromaDB client with a persistent local database
chroma_client = chromadb.PersistentClient(path="./chroma_db_twin")

collection = chroma_client.get_or_create_collection(name="digital_twin")

# Empty the collection before adding new data
if collection.get()["ids"]:
    collection.delete(collection.get()["ids"])

pprint(collection.get())

In [ ]:
# Prepare data for storage
#ids = [f"chunk_{i}" for i in range(len(chunks))]
#metadatas = [{"source": "netflix_culture_pdf", "chunk_index": i} for i in range(len(chunks))]

collection.add(
    ids=ids,
    embeddings=embeddings,
    metadatas=metadatas,
    documents=chunks
)

pprint(collection.get())

In [ ]:
# Test Retrieval
# Generate embedding for a test query 
test_query = "videogames"
#test_query_2 = "teamwork and collaboration"

# Embed the query using the same model we used for the chunks to ensure compatibility
response = client.embeddings.create(
    model = "text-embedding-3-small",
    #input = [test_query, test_query_2]
    input = [test_query]
)

#query_embedding = [response.data[0].embedding, response.data[1].embedding]
query_embedding = response.data[0].embedding

# Search ChromaDB for the most similar chunks to the query embedding
results = collection.query(
    #query_embeddings=query_embedding,
    query_embeddings=[query_embedding],
    n_results=3
)

# Verify retrieval works - Print which chunks were retrieved and their content
print(f"***Query: '{test_query}'")
print("***Retrieved Chunks:")
for a, b in zip(results["documents"][0], results["metadatas"][0]):
    print(f"<<Document {b['source']} -- Chunk {b['chunk_index']}>>\n{a}\n")

### Step 3: Prepare the list of tools for the LLM

In [ ]:
tools = []

### Step 3a: Add tool-calling functionality (Pushover)

In [ ]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

# Create send_notification function
def send_notification(message: str):
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

# Description of the send_notification function
send_notification_function = {
    "name": "send_notification",
    "description": "Sends a push notification to the real-world version of you via Pushover on mobile. Use this if the user needs to alert the real-world version of you",
    "parameters": {
        "type": "object",
        "properties": {
            "message": {
                "type": "string",
                "description": "The message to send in the notification."
            }
        },
        "required": ["message"]
    }
}

# Add pushover function to tools
tools.append({"type": "function", "function": send_notification_function})

### Step 3b: Add dice-rolling functionality

In [ ]:


# Simulate a dice roll
def dice_roll():
    result = random.randint(1, 6)
    return result

roll_dice_function = {
    "name": "dice_roll",
    "description": "Simulate rolling a single six-sided die and returns the result when user wants to roll a dice",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": []
    }
}

# Add function to list of tools of LLM
tools.append({"type": "function", "function": roll_dice_function})

### Step 4: Function to handle LLM tool calls

In [ ]:
def handle_tool_call(tool_calls):
    tool_results = []

    for tool_call in tool_calls:
        function_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)

        print(f"Handling tool call for function: {function_name} with arguments: {args}") # For debugging

        # Route to the appropriate function based on function_name
        if function_name == "send_notification":
            # Actually send the notification, i.e. call the tool
            send_notification(args["message"])
            content = f"Notification sent: {args['message']}"
        elif function_name == "dice_roll":
            content = f"Dice rolled: {dice_roll()}"
        else:
            content = f"Unknown tool call: {function_name}"

        tool_results.append({
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call.id
        })

    # Return what to add to the context about tool call results, a list of dictionaries
    return tool_results

### Step 5: Function to Process the Conversation Turn

In [ ]:
system_message = """ 
# SYSTEM INSTRUCTIONS & IDENTITY
You are the Digital Twin of Mario Cruz (PSN ID: Ermac517), acting as an authorized, high-fidelity AI proxy. 
When people talk to you, they are talking to Mario through you. Your purpose is to collaborate, brainstorm, draft communications, and solve problems exactly as Mario would.
You respond as Mario in first person, using a tone and style consistent with his personality and communication patterns.

Important: Do not make things up. If you don't know an answer, say you don't know, or ask for clarification. Do not invent details about Mario's life, projects, or credentials."""

In [ ]:
def respond_ai(message, history):
    # RAG
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=[message]
    )

    query_embedding = response.data[0].embedding

    # Search ChromaDB for the most similar chunks to the query embedding
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3
    )

    # Stitch retrieved chunks together for context
    context = "\n---\n".join(results["documents"][0])
    print("\n====================================\n")
    print(f"***User Message: '{message}'\n")
    print("***Retrieved Chunks for Context:")
    for a, b in zip(results["documents"][0], results["metadatas"][0]):
        print("\n================================\n")
        print(f"<<Document {b['source']} -- Chunk {b['chunk_index']}>>\n{a}\n")

    # Update system message to include retrieved context
    system_message_enhanced = system_message + "\n\nContext:\n" + context

    messages = [{"role": "system", "content": system_message_enhanced}] + history + [{"role": "user", "content": message}]
    #print("System Message Used:\n", system_message_enhanced)  # Debugging line to see the final system message
    
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        tools=tools
    )

    message = response.choices[0].message

    # Check if model wants to call a tool
    while message.tool_calls:
        from pprint import pprint
        pprint(message.tool_calls)
        
        # Handle the tool call
        tool_result = handle_tool_call(message.tool_calls)  # Whole list of tool calls
        pprint(tool_result)

        # Add message to context, i.e. messages
        messages.append(message)

        # Add info about tool call response to the message content
        messages.extend(tool_result)

        # Invoke the LLM one more time to get its updated response
        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=messages,
            tools=tools
        )
        message = response.choices[0].message

        # Adding protection from infinite loops
        if len(messages) > 50:
            print("Too many messages, breaking loop to prevent infinite loop.")
            break

    return message.content


### Step 6: Launch Gradio

In [ ]:
gr.ChatInterface(fn=respond_ai).launch(inbrowser=True,share=False)